## Skieur Mapping Analysis: MP1 vs MP2

Two data-loading methods. Pick one.

| Method | Source | Data | Mapping label |
|--------|--------|------|---------------|
| **A** | NAS raw `.npy` | `tracking_only` + `mapping_change` merged | `Mapping_Type` column in features |
| **B** | Local pickles | `playback` + `mapping_change_only` | `mapping_label` in `build_lmm_dataframe` |

**Method A advantages**: same neurons, same recording, Mapping_Type is a genuine within-session factor.
**Method B advantages**: spike-sorted, simpler pipeline, already proven in `Skieur_Full_LMM.ipynb`.


In [ ]:
# ============================================================
# Core imports + config
# ============================================================
import os, sys, pickle, gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, ttest_ind, chi2
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm

from utils_load_data import *
from utils_trajectories import *

# ---- Paths ----
save_directory = "./"

# ---- Parameters ----
dt = 0.005
t_pre, t_post = 0.3, 0.3
time = np.arange(-t_pre, t_post + dt, dt)
n_pre = int(t_pre/dt - 1)
NAS = r"\\129.199.81.18\\data5\\eTheremin"

# ---- Plotting style ----
mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

print("Imports and config ready.")


### Method A: Raw NAS `.npy` with merged MP1+MP2

Loads `data_0.005.npy` + `features_0.005.npy` from NAS.
Pairs `tracking_only` (no PB) + `mapping_change` (has PB) sessions,
concatenates along time, labels each row with `Mapping_Type`.

Data is already baseline-removed + smoothed by `get_data_from_nas`.


In [ ]:
# ============================================================
# Method A: get_data_from_nas
# ============================================================

def get_data_from_nas(all_sessions_path, dt, expect_pairs=True, gc_or_not=True):
    """Load raw .npy data from NAS, pair MP1+MP2, compute features.

    Returns: n_data_s, f_data_s
      - n_data_s[i] = (neurons, time)  already baseline-removed + smoothed
      - f_data_s[i] = DataFrame with Mapping_Type, Position, Speed_x, etc.
    """
    n_data_s, f_data_s = [], []
    global_id = 0

    unique_tones_sorted = np.sort(np.linspace(1000, 4000, 30))
    pixels_sorted = np.linspace(0, 28, len(unique_tones_sorted))
    speed_bins = [0, 0.01, 1, 2, 3, 4, 5, 6, np.inf]

    def process_and_store(n_mat, f_df, current_id):
        for col in ['Mock_frequency', 'Played_frequency']:
            if col not in f_df.columns:
                f_df[col] = np.nan

        positional_freq = np.where(
            f_df['Condition'].isin([0, -1]),
            f_df['Played_frequency'],
            f_df['Mock_frequency']
        )
        positional_freq = np.nan_to_num(positional_freq, nan=unique_tones_sorted[0])
        positions = np.interp(positional_freq, unique_tones_sorted, pixels_sorted)
        f_df['Position'] = gaussian_filter(positions, sigma=10)
        f_df['Speed_x'] = np.abs(np.append(0, np.diff(f_df['Position']))) * 100

        n_mat_o = n_mat - n_mat.mean(axis=1, keepdims=True)
        n_mat_smooth = gaussian_filter(n_mat_o, sigma=1, axes=1)

        n_data_s.append(n_mat_smooth)
        f_data_s.append(f_df)
        return current_id + len(n_mat_smooth)

    all_sessions_path = sorted(all_sessions_path)
    pending_original = None

    for file in tqdm(all_sessions_path, desc='Loading NAS'):
        session_name = os.path.basename(file)
        try:
            headstage_path = os.path.join(file, 'headstage_0')
            n_data = np.load(os.path.join(headstage_path, f'data_{dt}.npy'))
            f_raw = np.load(os.path.join(headstage_path, f'features_{dt}.npy'), allow_pickle=True)
            f_data = pd.DataFrame(list(f_raw))

            if gc_or_not:
                gc_idx = np.load(os.path.join(headstage_path, 'good_clusters.npy'))
            else:
                gc_idx = np.arange(len(n_data))
            n_data = n_data[gc_idx, :].astype(float)

            has_pb = False
            if 'Condition' in f_data.columns:
                unique_conds = set(f_data['Condition'].dropna().unique())
                if any(c in unique_conds for c in [1, 1.0]):
                    has_pb = True

            if not has_pb and 'Condition' in f_data.columns:
                f_data['Condition'] = f_data['Condition'].replace({-1.0: 0.0})

            if not expect_pairs:
                f_data['Mapping_Type'] = 'Standalone'
                global_id = process_and_store(n_data, f_data, global_id)
            else:
                if not has_pb:
                    f_data['Mapping_Type'] = 'MP1 (tracking_only)'
                    if pending_original is not None:
                        print(f"  WARNING: consecutive MP1; saving orphan [{pending_original['name']}]")
                        global_id = process_and_store(
                            pending_original['n_data'], pending_original['f_data'], global_id)
                    pending_original = {'name': session_name, 'n_data': n_data, 'f_data': f_data}
                else:
                    f_data['Mapping_Type'] = 'MP2 (mapping_change)'
                    if pending_original is not None:
                        n_merged = np.concatenate((pending_original['n_data'], n_data), axis=1)
                        f_merged = pd.concat([pending_original['f_data'], f_data], ignore_index=True)
                        print(f"  Merged: [{pending_original['name']}] + [{session_name}] "
                              f"-> {n_merged.shape[1]} tp, {n_merged.shape[0]} neurons")
                        global_id = process_and_store(n_merged, f_merged, global_id)
                        pending_original = None
                    else:
                        print(f"  WARNING: orphan MP2 [{session_name}]; saving standalone")
                        global_id = process_and_store(n_data, f_data, global_id)
        except Exception as e:
            print(f"  ERROR {session_name}: {e}")

    if pending_original is not None:
        print(f"  WARNING: final orphan MP1 [{pending_original['name']}]; saving standalone")
        global_id = process_and_store(
            pending_original['n_data'], pending_original['f_data'], global_id)

    return n_data_s, f_data_s


print("get_data_from_nas ready.")


### Method B: Pre-saved spike-sorted pickles

Loads `{prefix}_{session_type}_{dt}_data_ss` pickles from NAS or local.
MP1 = playback (both TR+PB), MP2 = mapping_change_only.
`mapping_label` is passed to `build_lmm_dataframe` at analysis time.


In [ ]:
# ============================================================
# Method B: load_pickled_ss
# ============================================================

def load_pickled_ss(file_prefix, session_type, dt):
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data_ss")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature_ss")
    with open(data_path, "rb") as f:
        n_data = pickle.load(f)
    with open(feat_path, "rb") as f:
        f_data = pickle.load(f)
    return n_data, f_data


print("load_pickled_ss ready.")


In [ ]:
USE_METHOD = 'B'  # 'A' = NAS raw .npy, 'B' = pre-saved SS pickles

# For Method B, choose which session types to load:
#   ('playback', 'mapping_change_only')  -> MP1 has TR+PB, conventional comparison
#   ('tracking_only', 'mapping_change_only') -> MP1 = TR only, MP2 = TR+PB
B_MP1_TYPE = 'tracking_only'         # session_type for MP1 pickle
B_MP2_TYPE = 'mapping_change_only'   # session_type for MP2 pickle

print(f"Using Method {USE_METHOD}")
if USE_METHOD == 'B':
    print(f"  MP1 = {B_MP1_TYPE}, MP2 = {B_MP2_TYPE}")


In [ ]:
# ============================================================
# Method A: Load + merge from NAS
# ============================================================
if USE_METHOD == 'A':
    nas_skieur = os.path.join(NAS, 'SKIEUR')
    all_dirs = sorted([
        os.path.join(nas_skieur, d)
        for d in os.listdir(nas_skieur)
        if os.path.isdir(os.path.join(nas_skieur, d))
        and os.path.exists(os.path.join(nas_skieur, d, 'headstage_0', f'data_{dt}.npy'))
    ])
    print(f"Found {len(all_dirs)} session dirs on NAS with data_{dt}.npy")
    for i, d in enumerate(all_dirs):
        print(f"  [{i:3d}] {os.path.basename(d)}")

    n_data_all, f_data_all = get_data_from_nas(all_dirs, dt, expect_pairs=True, gc_or_not=True)

    print(f"\nLoaded {len(n_data_all)} sessions (merged MP1+MP2)")
    for i, (nd, fd) in enumerate(zip(n_data_all, f_data_all)):
        mt = fd['Mapping_Type'].unique()
        conds = fd['Condition'].dropna().unique() if 'Condition' in fd.columns else []
        print(f"  Session {i}: {nd.shape[0]} neurons, {nd.shape[1]} tp, "
              f"Mapping={list(mt)}, Condition={list(conds)}")

    DATA_PREPROCESSED = True
else:
    print("Method A skipped (set USE_METHOD='A' to run).")


In [ ]:
# ============================================================
# Method B: Load from pre-saved SS pickles
# ============================================================
if USE_METHOD == 'B':
    mp1_label = 'MP1 (' + B_MP1_TYPE + ')'
    mp2_label = 'MP2 (' + B_MP2_TYPE + ')'

    print(f"Loading {mp1_label}...")
    n_data_mp1, f_data_mp1 = load_pickled_ss("SKIEUR_hs_0", B_MP1_TYPE, dt)
    print(f"  {mp1_label}: {len(n_data_mp1)} sessions")

    print(f"Loading {mp2_label}...")
    n_data_mp2, f_data_mp2 = load_pickled_ss("SKIEUR_hs_0", B_MP2_TYPE, dt)
    print(f"  {mp2_label}: {len(n_data_mp2)} sessions")

    FORCE_EQUAL = 1
    if FORCE_EQUAL:
        n_min = min(len(n_data_mp1), len(n_data_mp2))
        n_data_mp1 = n_data_mp1[:n_min]
        f_data_mp1 = f_data_mp1[:n_min]
        n_data_mp2 = n_data_mp2[:n_min]
        f_data_mp2 = f_data_mp2[:n_min]
        print(f"FORCE_EQUAL=1: trimmed to {n_min} sessions each")

    # Show condition breakdown for each MP
    for label, f_list in [(mp1_label, f_data_mp1), (mp2_label, f_data_mp2)]:
        all_conds = set()
        for fd in f_list:
            if 'Condition' in fd.columns:
                all_conds.update(fd['Condition'].dropna().unique())
        print(f"  {label} Condition values: {sorted(all_conds)}")

    DATA_PREPROCESSED = False
else:
    print("Method B skipped (set USE_METHOD='B' to run).")


---
## Build Trajectories

Extracts H1/H2 PSTH trajectories, baseline-subtracted, averaged across triggers.
For Method A, data is already preprocessed; for Method B, applies `smooth_data` + `remove_average`.


In [ ]:
# ============================================================
# Build trajectories
# ============================================================

if USE_METHOD == 'B':
    n_data_mp1_proc = remove_average(smooth_data(n_data_mp1))
    n_data_mp2_proc = remove_average(smooth_data(n_data_mp2))

    data_pairs = [
        (mp1_label, (n_data_mp1_proc, f_data_mp1)),
        (mp2_label, (n_data_mp2_proc, f_data_mp2)),
    ]
    traj_data = {}
    for mp_label, (n_data_list, f_data_list) in data_pairs:
        n_data_list, f_data_list = re_organise_data([n_data_list], [f_data_list])
        n_before = len(n_data_list)
        n_data_list, f_data_list = zip(*[(n, f) for n, f in zip(n_data_list, f_data_list)
                                          if abs(n.shape[-1] - len(f)) <= 2])
        n_data_list, f_data_list = list(n_data_list), list(f_data_list)
        n_dropped = n_before - len(n_data_list)
        if n_dropped > 0:
            print(f"  {mp_label}: shape-filter dropped {n_dropped} of {n_before} sessions")

        n_half = len(n_data_list) // 2
        beg_n, exp_n = list(n_data_list[:n_half]), list(n_data_list[n_half:])
        beg_f, exp_f = list(f_data_list[:n_half]), list(f_data_list[n_half:])

        traj = {}
        for q in [(0.0, 0.5), (0.5, 1.0)]:
            r_beg = extract_traj_subset(
                beg_n, beg_f, t_pre, t_post, dt,
                overlap_thresh=1.0, n_pre=n_pre, full=True,
                trial_start=q[0], trial_end=q[1])
            r_exp = extract_traj_subset(
                exp_n, exp_f, t_pre, t_post, dt,
                overlap_thresh=1.0, n_pre=n_pre, full=True,
                trial_start=q[0], trial_end=q[1])
            tb, pb, *_ = r_beg
            te, pe, *_ = r_exp
            traj[q] = {'track': {'beg': tb, 'exp': te},
                       'pb':    {'beg': pb, 'exp': pe}}
        traj_data[mp_label] = traj
        print(f"  {mp_label}: {n_half*2} sessions -> {len(tb)} TR + {len(pb)} PB traj")

    # For downstream compatibility with Skieur_Full_LMM cells
    traj_by_half_mp1 = traj_data[mp1_label]
    traj_by_half_mp2 = traj_data[mp2_label]
    print(f"\\ntraj_by_half_mp1 ({mp1_label}), traj_by_half_mp2 ({mp2_label}) ready for LMM.")

elif USE_METHOD == 'A':
    # Method A: data preprocessed, split by Mapping_Type
    traj_data = {}
    for mapping_label in ['MP1 (tracking_only)', 'MP2 (mapping_change)']:
        traj = {}
        for q in [(0.0, 0.5), (0.5, 1.0)]:
            track_list, pb_list = [], []
            for nd, fd in zip(n_data_all, f_data_all):
                mask = fd['Mapping_Type'] == mapping_label
                if not mask.any():
                    continue
                fd_sub = fd[mask].reset_index(drop=True)
                r = extract_traj_subset(
                    [nd], [fd_sub], t_pre, t_post, dt,
                    overlap_thresh=1.0, n_pre=None, full=True,
                    trial_start=q[0], trial_end=q[1])
                track_list.extend(r[0])
                pb_list.extend(r[1])
            traj[q] = {'track': track_list, 'pb': pb_list}
        traj_data[mapping_label] = traj

    n_tr = sum(1 for t in traj_data['MP1 (tracking_only)'][(0.0, 0.5)]['track'] if t is not None)
    n_pb = sum(1 for t in traj_data['MP2 (mapping_change)'][(0.0, 0.5)]['pb'] if t is not None)
    print(f"Method A trajectories: {n_tr} TR (MP1), {n_pb} PB (MP2)")

else:
    print("Set USE_METHOD first.")


In [ ]:
# ============================================================
# Session list: Beginner/Expert split
# ============================================================
import pandas as pd
sheet_url = ("https://docs.google.com/spreadsheets/d/"
             "1sFatSTXO0j3OONKstz7YN-mM04kNMjk_r7zo951yicU/"
             "gviz/tq?tqx=out:csv&sheet=SKIEUR")
df_sheet = pd.read_csv(sheet_url)
df_sheet = df_sheet[df_sheet["use"] == "yes"]

if USE_METHOD == 'B':
    for mp_label, n_data_list, stype in [
        (mp1_label, n_data_mp1, B_MP1_TYPE),
        (mp2_label, n_data_mp2, B_MP2_TYPE),
    ]:
        n_sess = len(n_data_list)
        n_half = n_sess // 2
        sessions = df_sheet[df_sheet["type"] == stype]["session"].tolist()[:n_sess]
        print(f"  {mp_label}: {n_sess} sessions")
        print(f"    Beginner ({n_half}): {sessions[:n_half]}")
        print(f"    Expert   ({n_sess - n_half}): {sessions[n_half:]}")
        print()

elif USE_METHOD == 'A':
    for stype in ['tracking_only', 'mapping_change']:
        sessions = df_sheet[df_sheet["type"] == stype]["session"].tolist()
        print(f"  {stype}: {len(sessions)} sessions in Google Sheet")
    print("  (Method A pairs them automatically via get_data_from_nas)")

print("\\nReady for LMM.")


---
## Quick Tests + LMM

The cells below mirror `Skieur_Full_LMM.ipynb` sections 2.1–2.5.
Run them in order. They use `traj_by_half_mp1` and `traj_by_half_mp2`
defined above by Method B (or adapt for Method A).


In [ ]:
# ===============================================================
# Quick tests: Track vs PB, MP1 vs MP2
# ===============================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import build_lmm_dataframe
from scipy.stats import ttest_rel, ttest_ind

all_p_vals_anova = []
all_p_labels_anova = []

for metric in ["mean", "peak"]:
    for t_window, label in [(0.1, "0-100ms"), (0.3, "0-300ms")]:
        sep = "=" * 60
        print()
        print(sep)
        print("  METRIC: " + metric + "  |  WINDOW: " + label)
        print(sep)

        df1 = build_lmm_dataframe(traj_by_half_mp1, "MP1", time,
                                   auc_t_start=0.0, auc_t_end=t_window,
                                   split_half=False, metric=metric)
        df2 = build_lmm_dataframe(traj_by_half_mp2, "MP2", time,
                                   auc_t_start=0.0, auc_t_end=t_window,
                                   split_half=False, metric=metric)
        df = pd.concat([df1, df2], ignore_index=True)

        # Track vs Playback (paired)
        tr = df[df["condition"] == "Track"]["response"].values
        pb = df[df["condition"] == "Playback"]["response"].values
        t_tr_pb, p_tr_pb = ttest_rel(pb, tr)
        print("  Track vs Playback (paired t-test):")
        print(f"    Track {tr.mean():.4f}, PB {pb.mean():.4f}, diff={pb.mean()-tr.mean():.4f}")
        print(f"    t={t_tr_pb:.3f}, p={p_tr_pb:.4e}")
        all_p_vals_anova.append(p_tr_pb)
        all_p_labels_anova.append("Track-vs-PB_" + metric + "_" + label)

        # MP1 vs MP2 (Welch)
        df_neuron = df.groupby(["neuron_id", "mapping"])["response"].mean().reset_index()
        mp1_vals = df_neuron[df_neuron["mapping"] == "MP1"]["response"].values
        mp2_vals = df_neuron[df_neuron["mapping"] == "MP2"]["response"].values
        t_mp, p_mp = ttest_ind(mp1_vals, mp2_vals, equal_var=False)
        print()
        print("  MP1 vs MP2 (Welch t-test):")
        print(f"    MP1 {np.mean(mp1_vals):.4f}, MP2 {np.mean(mp2_vals):.4f}")
        print(f"    t={t_mp:.3f}, p={p_mp:.4e}")
        all_p_vals_anova.append(p_mp)
        all_p_labels_anova.append("MP1-vs-MP2_" + metric + "_" + label)
        print()

# Global FDR
from statsmodels.stats.multitest import multipletests
if all_p_vals_anova:
    reject, p_fdr, _, _ = multipletests(all_p_vals_anova, method='fdr_bh')
    print("=" * 70)
    print("  QUICK TESTS: GLOBAL FDR CORRECTION (BH)")
    print("=" * 70)
    for label, p_raw, p_corr in zip(all_p_labels_anova, all_p_vals_anova, p_fdr):
        sig = '***' if p_corr<0.001 else ('**' if p_corr<0.01 else ('*' if p_corr<0.05 else 'n.s.'))
        print(f"  {label:<40s} raw={p_raw:.6f}  fdr={p_corr:.6f}  {sig}")


In [ ]:
# ================================================================
# Full LMM: condition * mapping * expertise * half
# ================================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import run_lmm_analysis, compare_random_effect_structures

for metric in ["mean", "peak"]:
    for t_window, label in [(0.1, "0-100ms"), (0.3, "0-300ms")]:
        sep = "=" * 70
        print()
        print(sep)
        print("  METRIC: " + metric + "  |  WINDOW: " + label)
        print(sep)
        result_lmm, df_lmm = run_lmm_analysis(
            traj_by_half_mp1, traj_by_half_mp2, time, save_directory,
            auc_t_start=0.0, auc_t_end=t_window, metric=metric
        )
        compare_random_effect_structures(df_lmm)


In [ ]:
# Peak 0-200ms: Full LMM
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import run_lmm_analysis, compare_random_effect_structures

result_peak, df_peak = run_lmm_analysis(
    traj_by_half_mp1, traj_by_half_mp2, time, save_directory,
    auc_t_start=0.0, auc_t_end=0.2, metric="peak"
)
compare_random_effect_structures(df_peak)
